# Project 2 Migration & Education

## Part 1- Introduction

In the project, I explore whether world regions with a higher share of international migrants also tend to have higher upper secondary enrollment rates.

I focus on six major world regions in 2015:
- Africa
- Northern America
- Latin America & the Caribbean
- Europe
- Asia
- Oceania

The data are taken from two tables in the United Nations: 

- International migrants as a percentage of the toal population, by region (2015)
- Gross enrollment ratio in upper secondary education, by region (2015)

Tp keep the analysis simple and transparnet, I manually construct a small dataset in Python based on the official UN values. I then visualize the relationship between:
- **International migrants (% of population)**
- **Upper secondary gross enrollment ratio (%)**

The goal is not to show causality, but to see whether a broad pattern emerges between migration intensity and education outcomes across world regions.

## Part 2- Clean and Merge the Datasets

### Step 1: Load the two libraries and datasets
#### Update the filenames to match your downloaded CSVs

In [253]:
import pandas as pd
migration_2015 = pd.read_csv("SYB67_327_202411_International Migrants and Refugees.csv")
education_2015 = pd.read_csv("SYB67_309_202411_Education.csv")


#### Inspect the datasets to understand their structures
Explain:
- What each column means
- Any issues (extrea rows, footnotes, strange formatting)

In [255]:
migration_2015.head()

,T04,International migrants and refugees,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6
0,Region/Country/Area,NaN,Year,Series,Value,Footnotes,Source
1,1,"Total, all countries or areas",2005,International migrant stock: Both sexes (number),"191,446,828",NaN,"United Nations Population Division, New York, ..."
2,1,"Total, all countries or areas",2005,International migrant stock: Both sexes (% tot...,2.9,NaN,"United Nations Population Division, New York, ..."
3,1,"Total, all countries or areas",2005,International migrant stock: Male (% total Pop...,3.0,NaN,"United Nations Population Division, New York, ..."
4,1,"Total, all countries or areas",2005,International migrant stock: Female (% total P...,2.9,NaN,"United Nations Population Division, New York, ..."


In [256]:
education_2015.head()

,T07,"Enrollment in primary, lower secondary and upper secondary education levels",Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6
0,Region/Country/Area,NaN,Year,Series,Value,Footnotes,Source
1,1,"Total, all countries or areas",2005,Students enrolled in primary education (thousa...,"680,726",Estimate.,"United Nations Educational, Scientific and Cul..."
2,1,"Total, all countries or areas",2005,Gross enrollment ratio - Primary (male),104.5,Estimate.,"United Nations Educational, Scientific and Cul..."
3,1,"Total, all countries or areas",2005,Gross enrollment ratio - Primary (female),99.8,NaN,"United Nations Educational, Scientific and Cul..."
4,1,"Total, all countries or areas",2005,Students enrolled in lower secondary education...,"311,138",NaN,"United Nations Educational, Scientific and Cul..."


### Step 2: Clean & selected needed columns

#### Read and clean the Migration data

In [257]:
migration_2015 = migration_2015.rename(columns={
    "International migrants and refugees": "region",
    "Unnamed: 3": "year",
    "Unnamed: 4": "series",
    "Unnamed: 5": "value"
})

# Remove the row that contains "Region/Country/Area"
migration_2015 = migration_2015.iloc[1:].reset_index(drop=True)

# Make columns numeric safely
migration_2015["year"] = pd.to_numeric(migration_2015["year"], errors="coerce")
migration_2015["value"] = pd.to_numeric(migration_2015["value"].astype(str).str.replace(",", ""), errors="coerce")


#### Read and clean the Education data

In [258]:
education_2015 = education_2015.rename(columns={
    "Enrollment in primary, lower secondary and upper secondary education levels": "region",
    "Unnamed: 2": "year",
    "Unnamed: 3": "series",
    "Unnamed: 4": "value"
})

education_2015 = education_2015.iloc[1:].reset_index(drop=True)

education_2015["year"] = pd.to_numeric(education_2015["year"], errors="coerce")
education_2015["value"] = pd.to_numeric(education_2015["value"].astype(str).str.replace(",", ""), errors="coerce")

print(education_2015.head())
print(education_2015.columns.tolist())


  T07                         region  year  \
0   1  Total, all countries or areas  2005   
1   1  Total, all countries or areas  2005   
2   1  Total, all countries or areas  2005   
3   1  Total, all countries or areas  2005   
4   1  Total, all countries or areas  2005   

                                              series     value Unnamed: 5  \
0  Students enrolled in primary education (thousa...  680726.0  Estimate.   
1            Gross enrollment ratio - Primary (male)     104.5  Estimate.   
2          Gross enrollment ratio - Primary (female)      99.8        NaN   
3  Students enrolled in lower secondary education...  311138.0        NaN   
4  Gross enrollment ratio - Lower secondary level...      80.7        NaN   

                                          Unnamed: 6  
0  United Nations Educational, Scientific and Cul...  
1  United Nations Educational, Scientific and Cul...  
2  United Nations Educational, Scientific and Cul...  
3  United Nations Educational, Scientifi

### Step 3: Filter to 2015 and the six regions

Conceptually, the data come from two separate UN tables:
1. A migration table with "region", "year", and "migrant_pct"
2. An education table with "region", "year", and "uppersec_GER"

To illustrate this structure in code, I first split the combined dataset into 2 smaller dataframes

In [261]:
# Create a "migration" dataset
migration_2015 = combined_2015[["region", "year", "migrant_pct"]].copy()

# Create an "education" dataset
education_2015 = combined_2015[["region", "year", "uppersec_GER"]].copy()

print("Migration dataset:")
display(migration_2015)

print("Education dataset:")
display(education_2015)


Migration dataset:


,region,year,migrant_pct
0,Africa,2015,2.0
1,Northern America,2015,15.3
2,Latin America & the Caribbean,2015,1.8
3,Europe,2015,10.1
4,Asia,2015,1.7
5,Oceania,2015,20.3


Education dataset:


,region,year,uppersec_GER
0,Africa,2015,37.6
1,Northern America,2015,100.5
2,Latin America & the Caribbean,2015,74.1
3,Europe,2015,112.8
4,Asia,2015,67.1
5,Oceania,2015,117.8


### Merging the datasets
Both datasets share the same keys: "region" and "year". This step replicates the process of merging two separate UN tables that share the same regional and temporal structure.

In [262]:
merged_2015 = pd.merge(
    migration_2015,
    education_2015,
    on=["region", "year"],
    how="inner"
)

merged_2015


,region,year,migrant_pct,uppersec_GER
0,Africa,2015,2.0,37.6
1,Northern America,2015,15.3,100.5
2,Latin America & the Caribbean,2015,1.8,74.1
3,Europe,2015,10.1,112.8
4,Asia,2015,1.7,67.1
5,Oceania,2015,20.3,117.8


## Part 3- Visualization

### To explore the relationship between migration and education, I plot:
- ***X-axis***: international migrants as a percentage of the population ("migrant_pct")
- ***Y-axis***: upper secondary gross enrollment ratio ("uppersec_GER")

Each point represents a world region, and is labeled by region name.

In [263]:
import plotly.express as px

fig = px.scatter(
    combined_2015,
    x="migrant_pct",
    y="uppersec_GER",
    text="region",
    labels={
        "migrant_pct": "International migrants (% of population), 2015",
        "uppersec_GER": "Upper secondary gross enrollment ratio (%), 2015"
    },
    title="Migration vs Upper Secondary Enrollment by Region (2015)"
)

fig.update_traces(textposition="top center")
fig.show()


## Part 4- Finding & Interpretation

The scatterplot suggests a positive association between international migration and upper secondary enrollment across world regions:

- Regions with **higher shares of international migrants**, such as **Northern America**, **Europe**, and **Oceania**, also show **very high upper secondary gross enrollment ratios**, which is around or above 100%.
- Regions with **low migrant shares of international migrants**, such as **Africa** display **lower upper secondary enrollment**, particularly in Africa is extremly low estimated 38%.
- **Latin America & the Caribbean** and **Asia** occupies a middle position in between low migrant share but moderately high upper secondary gross enrollment ratio.

This pattern does not imply that migration cause higher enrollment. Rather, both migration and education outcomes are likely driven by underlying factors such as:
- overall level of economic development
- state and public investment in education
- Demographic structure and labor market demand
- Historical patterns of migration
- Families

In broad terms, regions that are attractive destinations for migrant also tend to have stronger education systems,while regions with weaker education outcomes are oftern net senders rather than receivers of migrants.